# 極座標のホッジラプラシアン

このNotebookでは、計量と微分形式の演算子からスカラーに対するホッジラプラシアンを
導出します。$r>0$ の極座標 $(r,\theta)$ では、

$$g=dr^2+r^2d\theta^2.$$

となります。以下で採用する余微分の規約では、結果は通常の座標表示による
正符号のラプラス作用素に負号を付けたものになります。


## 極座標の計量

共変計量と反変計量の両方にテンソルの添字を明示します。これによりEgisonは、
ホッジスターの公式に現れる縮約を自動的に処理できます。


In [1]:
declare symbol r, θ : MathValue

def N : Integer := 2
def x : Vector MathValue := [| r, θ |]

def g_i_j : Matrix MathValue :=
  [| [| 1, 0 |], [| 0, r ^ 2 |] |]_i_j
def g~i~j : Matrix MathValue :=
  [| [| 1, 0 |], [| 0, r ^ (-2) |] |]~i~j


In [2]:
g_#_#


$\begin{pmatrix} 1 & 0 \\ 0 & r^{2} \\ \end{pmatrix}_{\#\#}^{\;\;}$

## 外微分とホッジスター

ホッジスターは、レヴィ・チヴィタテンソル、逆計量、体積密度
$\sqrt{|g|}=r$ を組み合わせて定義されます。


In [3]:
def d (A : Tensor MathValue) : Tensor MathValue :=
  !(flip ∂/∂) x A

def hodge (A : Tensor MathValue) : Tensor MathValue :=
  let k := dfOrder A
   in withSymbols [i, j]
        sqrt (M.det g_#_#) *
        foldl
          (.)
          ((subrefs A (map 1#j_$1 (between 1 k))) .
           (subrefs (ε' N k) (map 1#i_$1 (between 1 N))))
          (map 1#g~(i_$1)~(j_$1) [1..k])


## 余微分とラプラシアン

余微分は $d$ の計量に関する随伴であり、二つのホッジスターを用いて表せます。
形式の次数を調べることで、0形式と最高次形式に適した端点の公式を選択します。


In [4]:
def δ (A : Tensor MathValue) : Tensor MathValue :=
  let k := dfOrder A
   in (-1) ^ (N * (k + 1) + 1) * (hodge (d (hodge A)))

def Δ (A : Tensor MathValue) : Tensor MathValue :=
  match (dfOrder A) as integer with
  | #0 -> δ (d A)
  | #N -> d (δ A)
  | _  -> d (δ A) + δ (d A)

def f : MathValue := function (r, θ)


任意のスカラー関数に作用素を適用し、導関数を記号のまま残すことで、
座標表示の公式を直接確認できます。


In [5]:
Δ f


$-\frac{\partial^2 f}{\partial 1^2} - \frac{\partial f}{\partial 1} r^{-1} - \frac{\partial^2 f}{\partial 2^2} r^{-2}$

表示される式は

$$
\Delta f=-\left(
  \frac{\partial^2f}{\partial r^2}
  +\frac1r\frac{\partial f}{\partial r}
  +\frac1{r^2}\frac{\partial^2f}{\partial\theta^2}
\right).
$$

です。$1/r$ と $1/r^2$ の項を手作業で加えているわけではありません。
これらはホッジスターに含まれる計量の行列式と逆計量から現れます。
